In [4]:

import scanpy as sc
import pandas as pd
cmc_dir='/sc/arion/projects/CommonMind/roussp01a/ENT/snRNAseq/'

N_adata_f = sc.read(cmc_dir+'qc_scanpy/ent_nn_merge_rawcount.h5ad')

meta_data = pd.read_csv('/sc/arion/projects/roussp01a/liting/Olf/data/ent_nn_merge_cca.metadata.csv', index_col=0)

N_adata_f = N_adata_f[meta_data.index]
N_adata_f.obs["N_types_stage"]= meta_data.loc[N_adata_f.obs.index,'cca_N_types_stage'] 

sc.pp.filter_genes(N_adata_f, min_cells=5)
sc.pp.normalize_total(N_adata_f, target_sum=1e4)
sc.pp.log1p(N_adata_f)
N_adata_f.obs['dataset_ntypes']=N_adata_f.obs['N_types_stage'].astype(str)+N_adata_f.obs['dataset'].astype(str)

wang2025_marker = pd.read_table('/sc/arion/projects/roussp01a/liting/data_tools/marker_Li_2025.txt')
wang2025_marker = wang2025_marker.groupby('Cell type')['Gene'].apply(list).to_dict()
wang2025_marker = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in wang2025_marker.items()]))
wang2025_marker=wang2025_marker.drop(columns="Cajal-Retzius cell")
wang2025_marker=wang2025_marker.drop(columns="Unknown")
for x in wang2025_marker.keys():
    sc.tl.score_genes(N_adata_f,wang2025_marker[x][wang2025_marker[x].isin(N_adata_f.var.index)][:50],score_name=x,ctrl_size=50)



In [5]:
mean_module_score = N_adata_f.obs.groupby('N_types_stage').agg(dict(zip(wang2025_marker.columns.tolist(),['mean']*len(wang2025_marker.columns.tolist()))))
mean_module_score.to_csv('/sc/arion/projects/roussp01a/liting/Olf/data/liwang_OSN_modulescore.txt')

In [6]:
liwang = sc.read('/sc/arion/projects/roussp01a/liting/Olf/data/liwang.h5ad')

In [7]:
lister = sc.read('/sc/arion/projects/CommonMind/aging/hui/files/lister_processed.h5ad')
sc.pp.normalize_total(lister, target_sum=1e4)
sc.pp.log1p(lister)
lister=lister[lister.obs.major_clust!='Poor-Quality']
lister.obs["cluster_stage"]= lister.obs.stage_id.astype(str) + lister.obs.major_clust.astype(str)
group_counts = lister.obs['cluster_stage'].value_counts()
groups_to_keep = group_counts[group_counts >= 300].index.tolist()
lister_filtered = lister[lister.obs['cluster_stage'].isin(groups_to_keep)]
sc.tl.rank_genes_groups(
    lister_filtered,
    groupby='major_clust',
    method='wilcoxon'
)

In [8]:
result = lister_filtered.uns['rank_genes_groups']
groups = result['names'].dtype.names

top_n = 200

module_score_cols = []

for cluster in groups:

    names = result['names'][cluster]
    logfc = result['logfoldchanges'][cluster]
    fdr = result['pvals_adj'][cluster]

    df = pd.DataFrame({
        'gene': names,
        'logfc': logfc,
        'fdr': fdr
    })


    df_filtered = df[(df['logfc'] > 0.5) & (df['fdr'] < 0.05)]
    df_filtered = df_filtered.sort_values('logfc', ascending=False)
    top_genes = df_filtered['gene'].head(top_n).tolist()

    genes_in_data = [g for g in top_genes if g in N_adata_f.var_names]
    
    # print(len(top_genes))

    print(f"Cluster {cluster}: {len(genes_in_data)} genes after filtering used in module score")

    if genes_in_data:
        score_name = f"{cluster}"
        sc.tl.score_genes(
            N_adata_f,
            gene_list=genes_in_data[0:50],
            score_name=score_name
        )
        module_score_cols.append(score_name)


print("Module scores computed:", module_score_cols)



In [9]:
grouped_means = N_adata_f.obs.groupby('N_types_stage')[module_score_cols].median()
grouped_means.to_csv('/sc/arion/projects/roussp01a/liting/Olf/data/lister_OSN_modulescore.txt')